In [ ]:
# Keep imports and data/output paths relative to the repository root.
import os
import sys
from pathlib import Path
from datetime import datetime
from uuid import uuid4

for repository_dir in (Path("."), Path(".."), Path("../.."), Path("../../..")):
    if (repository_dir / "src" / "use_croma.py").is_file():
        os.chdir(repository_dir)
        break
else:
    raise FileNotFoundError("Open this notebook from inside the repository.")
if "src" not in sys.path:
    sys.path.insert(0, "src")

EXPERIMENT_NAME = "croma_sen1floods11/UperNet_head_S1S2"
NOTEBOOK_PATH = "experiments/croma_sen1floods11/UperNet_head_S1S2/02C_croma_on_sen1floods11_test_UperNet_S1S2.ipynb"


In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms.functional as FT
import torch.nn.functional as F
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import csv
import random
import math
import json
import pandas as pd

from PIL import Image
from pathlib import Path
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models.segmentation.fcn import FCNHead
from use_croma import PretrainedCROMA

In [ ]:
from croma_upernet_pangaea import (
    UPerNetHead, filter_labeled_training_pairs, random_labeled_crop, PANGAEA_REVISION
)
from einops import rearrange
from segmentation_metrics import (binary_metrics, confusion_counts, METRIC_KEYS,
    metric_table, save_per_chip_metrics, clean_json, plot_metric_history)

CROP_SIZE = 120
IGNORE_INDEX = 255
SCHEMA = "croma_s1s2_upernet_pangaea_v1"
S2_BANDS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13]  # Exclude B10/cirrus.
# Same pinned PANGAEA dataset statistics as S1, selecting the optical channels.
PREPROCESSING = {
    "name": "pangaea_sen1floods11_optical_mean_std", "modality": "optical",
    "bands": S2_BANDS,
    "band_names": ["B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B9", "B11", "B12"],
    "mean": [1626.91600224, 1396.03470631, 1364.06118417, 1218.22847919,
             1466.07290663, 2386.90297537, 2845.61256277, 2622.95796892,
             3077.48221481, 486.87436782, 2030.64763024, 1179.16607221],
    "std": [700.17133846, 739.09452682, 735.2482388, 864.936695,
            776.8803358, 921.36834309, 1084.37346097, 1022.63418007,
            1196.44255318, 336.61105431, 980.87061347, 764.60836557],
    "nan_to_num": "numpy_default_before_normalization", "clip": None,
    "ignore_index": IGNORE_INDEX, "source_revision": PANGAEA_REVISION,
}

# Concatenate only for synchronized cropping/tiling; the model splits the sensors before encoding.
OPTICAL_PREPROCESSING = PREPROCESSING
PREPROCESSING = {
    "name": "pangaea_sen1floods11_joint_mean_std", "modality": "both",
    "channel_order": ["VV", "VH"] + OPTICAL_PREPROCESSING["band_names"],
    "mean": [-10.184408, -16.895273] + OPTICAL_PREPROCESSING["mean"],
    "std": [4.255339, 5.290568] + OPTICAL_PREPROCESSING["std"],
    "optical_bands": S2_BANDS, "nan_to_num": "numpy_default_before_normalization",
    "clip": None, "ignore_index": IGNORE_INDEX, "source_revision": PANGAEA_REVISION,
}

def sar_path(optical_path):
    return optical_path.parent.parent / "S1Hand" / optical_path.name.replace("_S2Hand.tif", "_S1Hand.tif")

def load_split_paths(csv_file, x_dir, y_dir):
    pairs = []
    with open(csv_file, newline="") as handle:
        for x_name, y_name in csv.reader(handle):
            chip = x_name.strip().removesuffix("_S1Hand.tif")
            if y_name.strip() != chip + "_LabelHand.tif":
                raise ValueError(f"Mismatched split row: {x_name}, {y_name}")
            x_path, y_path = Path(x_dir) / (chip + "_S2Hand.tif"), Path(y_dir) / y_name.strip()
            if not x_path.is_file() or not y_path.is_file():
                raise FileNotFoundError(f"Missing pair: {x_path}, {y_path}")
            with rasterio.open(x_path) as x, rasterio.open(y_path) as y:
                if (x.count != 13 or x.shape != y.shape or x.crs != y.crs
                        or not x.transform.almost_equals(y.transform, precision=1e-10)):
                    raise ValueError(f"Band/grid mismatch: {chip}")
            with rasterio.open(sar_path(x_path)) as sar, rasterio.open(y_path) as y:
                if (sar.count != 2 or sar.shape != y.shape or sar.crs != y.crs
                        or not sar.transform.almost_equals(y.transform, precision=1e-10)):
                    raise ValueError(f"SAR/optical/label grid mismatch: {chip}")
            pairs.append((x_path, y_path))
    return pairs

def load_image_pair(pair):
    with rasterio.open(pair[0]) as src:
        image = src.read(S2_BANDS, out_dtype="float32", masked=True).filled(np.nan)
    with rasterio.open(sar_path(pair[0])) as src:
        sar = src.read([1, 2], out_dtype="float32", masked=True).filled(np.nan)
    image = np.concatenate([sar, image], axis=0)
    with rasterio.open(pair[1]) as src:
        mask = src.read(1).astype(np.int64)
    return image, mask

def prepare_sample(data):
    raw, mask = data
    image = torch.as_tensor(np.nan_to_num(raw), dtype=torch.float32)
    mask = torch.as_tensor(mask, dtype=torch.long).clone()
    mask[mask == -1] = IGNORE_INDEX
    if image.ndim != 3 or image.shape[0] != 14 or image.shape[-2:] != mask.shape:
        raise ValueError("Expected aligned 14-channel SAR/optical image and label.")
    if not set(torch.unique(mask).tolist()).issubset({0, 1, IGNORE_INDEX}):
        raise ValueError("Unexpected label values.")
    mean = image.new_tensor(PREPROCESSING["mean"])[:, None, None]
    std = image.new_tensor(PREPROCESSING["std"])[:, None, None]
    image = (image - mean) / std
    if not torch.isfinite(image).all():
        raise ValueError("Nonfinite normalized multimodal input.")
    return image, mask

class Sen1Floods11Dataset(Dataset):
    def __init__(self, data_list, preprocess_func):
        self.data_list, self.preprocess_func = data_list, preprocess_func
    def __len__(self):
        return len(self.data_list)
    def __getitem__(self, index):
        return self.preprocess_func(load_image_pair(self.data_list[index]))

def audit_joint_pairs(pairs):
    rows = []
    for pair in pairs:
        raw, _ = load_image_pair(pair)
        valid = np.isfinite(raw).all(axis=0)
        rows.append({"image": str(pair[0]), "usable_pixel_fraction": float(valid.mean()),
                     "exclude_missing_joint": not bool(valid.any())})
    return pd.DataFrame(rows)

class CROMAUPerNet(nn.Module):
    def __init__(self, croma, head):
        super().__init__()
        self.croma = croma.requires_grad_(False)
        self.head = head
        self.croma.eval()
        if max(head.out_indices) >= len(croma.cross_encoder.layers):
            raise ValueError('Requested blocks exceed the joint encoder depth.')
        if head.config['in_channels'] != croma.encoder_dim:
            raise ValueError('Decoder channel count does not match CROMA.')

    def train(self, mode=True):
        super().train(mode)
        self.croma.eval()
        return self

    @torch.no_grad()
    def extract_features(self, images):
        p = self.croma.s1_encoder.patch_size
        height, width = images.shape[-2:]
        if images.shape[1] != 14 or height != width or height % p or (height // p)**2 != self.croma.num_patches:
            raise ValueError('Expected paired 14-channel tiles matching CROMA image_resolution.')
        bias = self.croma.attn_bias.to(device=images.device, dtype=images.dtype)
        # Full unimodal streams supply the pretrained joint cross-attention encoder.
        x = self.croma.s1_encoder(images[:, :2], bias)
        context = self.croma.s2_encoder(images[:, 2:], bias)
        features = []
        for index, (self_attention, cross_attention, ffn) in enumerate(self.croma.cross_encoder.layers):
            x = x + self_attention(x, bias)
            x = x + cross_attention(x, context, bias)
            x = x + ffn(x)
            if index in self.head.out_indices:
                selected = self.croma.cross_encoder.norm_out(x) if index == self.head.out_indices[-1] else x
                features.append(selected.transpose(1, 2).reshape(x.shape[0], x.shape[2], height // p, width // p))
            if index == self.head.out_indices[-1]:
                break
        return features

    def forward(self, images):
        logits = self.head(self.extract_features(images))
        return F.interpolate(logits, size=images.shape[-2:], mode='bilinear', align_corners=False)

def latest_completed_checkpoint(run_root):
    candidates = []
    for manifest in Path(run_root).glob('train_*/run.json'):
        metadata = json.loads(manifest.read_text(encoding='utf-8'))
        if metadata.get('schema') != SCHEMA:
            continue
        history_path = manifest.parent / 'croma_training_history.json'
        if not history_path.is_file():
            continue
        history = json.loads(history_path.read_text(encoding='utf-8'))
        checkpoint = manifest.parent / Path(metadata['checkpoint']).name
        if history.get('epochs_completed') == metadata.get('epochs') and checkpoint.is_file():
            candidates.append((metadata['started_at'], checkpoint))
    if not candidates:
        raise FileNotFoundError('No completed UPerNet run found. Run the training notebook first.')
    return max(candidates, key=lambda item: item[0])[1]


#### **First things first**


In [ ]:
model_name = "croma_large_upernet_pangaea_s1s2.pt"
# None selects the latest completed UPerNet run, using its best validation-mIoU checkpoint.
# Set a relative Path here only to evaluate a specific older run.
checkpoint_path = None
TILE_BATCH_SIZE = 8


In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

pretrained_path = "pretrained/CROMA_large.pt"

path_to_the_dataset = "dataset/sen1floods11_hand_labeled_only/HandLabeled/"

path_to_the_dataset_index = "dataset/sen1floods11_hand_labeled_only/flood_handlabeled/"

path_to_X = path_to_the_dataset + "S2Hand/"

path_to_Y = path_to_the_dataset + "LabelHand/"

path_to_permanent_water = path_to_the_dataset + "JRCWaterHand"

path_to_test_csv = path_to_the_dataset_index + "flood_test_data.csv"
path_to_test_bolivia_csv = path_to_the_dataset_index + "flood_bolivia_data.csv"

print(device)

#### **Functions**

In [ ]:
## Note here mIoU is not being computed!
def preprocess_test(data):
   
    return prepare_sample(data)

#### **Creating datasets**

In [ ]:

# Verify official split membership, counts, disjointness, and S1/S2/label alignment.
split_files = {"train": "flood_train_data.csv", "valid": "flood_valid_data.csv",
               "non_bol": "flood_test_data.csv", "bol": "flood_bolivia_data.csv"}
expected_counts = {"train": 252, "valid": 89, "non_bol": 90, "bol": 15}
all_splits = {name: load_split_paths(Path(path_to_the_dataset_index) / filename, path_to_X, path_to_Y)
              for name, filename in split_files.items()}
for name, pairs in all_splits.items():
    assert len(pairs) == expected_counts[name], (name, len(pairs))
    assert len({x.stem for x, _ in pairs}) == len(pairs), f"Duplicates in {name}"
for i, name in enumerate(all_splits):
    for other in list(all_splits)[i+1:]:
        assert not ({x.stem for x, _ in all_splits[name]} & {x.stem for x, _ in all_splits[other]})
print("Verified S1/S2/label grids and disjoint split counts:", {k: len(v) for k, v in all_splits.items()})

test_data_non_bolivia = all_splits["non_bol"]
test_data_bolivia = all_splits["bol"]


In [ ]:
print(f"Non bolivia test samples: {len(test_data_non_bolivia)}")
print(f"Bolivia test samples: {len(test_data_bolivia)}")

In [ ]:
print(test_data_non_bolivia[0])
print(test_data_bolivia[0])

In [ ]:
test_dataset_non_bolivia = Sen1Floods11Dataset(
 data_list = test_data_non_bolivia,
 preprocess_func=preprocess_test
)

test_dataset_bolivia = Sen1Floods11Dataset(
 data_list = test_data_bolivia,
 preprocess_func=preprocess_test
)

In [ ]:
test_loader_non_bolivia = DataLoader(
    test_dataset_non_bolivia,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader_bolivia = DataLoader(
    test_dataset_bolivia,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

#### **Model**

In [ ]:
croma = PretrainedCROMA(pretrained_path= pretrained_path, 
                        size='large', 
                        modality='both', 
                        image_resolution=120)

croma = croma.to(device)

In [ ]:
croma.requires_grad_(False) # Freeze the weights of CROMA
croma.eval()

#### **Creating the head**

In [ ]:
if checkpoint_path is None:
    checkpoint_path = latest_completed_checkpoint(Path("runs") / EXPERIMENT_NAME)
checkpoint_path = Path(checkpoint_path)
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
if checkpoint.get("schema") != SCHEMA:
    raise ValueError("This notebook requires a PANGAEA-aligned multimodal UPerNet checkpoint.")
if checkpoint.get("preprocessing") != PREPROCESSING:
    raise ValueError("Checkpoint preprocessing does not match this notebook.")
if checkpoint.get("crop_size") != CROP_SIZE:
    raise ValueError("Checkpoint crop size does not match this notebook.")
if checkpoint.get("pretrained_weights") != pretrained_path:
    raise ValueError("Checkpoint requires different pretrained CROMA weights.")
head = UPerNetHead(**checkpoint["head_config"]).to(device)
print(f"Selected checkpoint: {checkpoint_path}")


In [ ]:
head.load_state_dict(checkpoint["head_state_dict"], strict=True)
print(f"UPerNet weights loaded: epoch {checkpoint['epoch']}, "
      f"validation mIoU {checkpoint['best_val_miou']:.4f}")


In [ ]:
model = CROMAUPerNet(croma, head).to(device)
model.eval()


In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)


In [ ]:
@torch.inference_mode()
def sliding_window_logits(
    model,
    image,
    device,
    tile_size=120,
    stride=None,
    tile_batch_size=TILE_BATCH_SIZE
):
    """
    Predict segmentation logits for one complete image.

    Parameters
    ----------
    model:
        Frozen CROMA + trained UPerNet head.

    image:
        Tensor [14, H, W], already normalized.

    device:
        CUDA or CPU.

    tile_size:
        Spatial size of each CROMA input (120).

    stride:
        None uses the PANGAEA evenly spaced sliding-window grid.

    tile_batch_size:
        Number of windows processed simultaneously.

    Returns
    -------
    logits:
        Tensor [2, H, W] on CPU.
    """

    model.eval()

    C, H, W = image.shape

    assert C == 14
    assert H >= tile_size and W >= tile_size
    assert stride is None or 0 < stride <= tile_size

    # Generate starting positions along one dimension
    def get_starts(length):
        import math
        # Upstream default: 5 evenly spaced windows per axis for 512px / 120px.
        count = math.ceil(length / tile_size) if stride is None else math.ceil((length - tile_size) / stride) + 1
        return torch.linspace(0, length - tile_size, count).round().long().tolist()

    # All window coordinates
    positions = [
        (top, left)
        for top in get_starts(H)
        for left in get_starts(W)
    ]

    # Accumulate predictions and coverage counts
    logits_sum = torch.zeros((2, H, W), dtype=torch.float32)
    counts = torch.zeros((H, W), dtype=torch.float32)

    # Process windows in batches
    for start in range(0, len(positions), tile_batch_size):

        batch_positions = positions[start:start + tile_batch_size]

        # Extract tiles
        tiles = torch.stack([
            image[:, top:top + tile_size, left:left + tile_size]
            for top, left in batch_positions
        ])

        tiles = tiles.to(device)

        # Run frozen CROMA + UPerNet head
        outputs = model(tiles).cpu()

        # Place tile predictions back into the full image
        for (top, left), logits in zip(batch_positions, outputs):

            logits_sum[
                :, top:top + tile_size, left:left + tile_size
            ] += logits

            counts[
                top:top + tile_size, left:left + tile_size
            ] += 1

    assert (counts > 0).all(), "Some pixels were not covered."

    # Average logits wherever windows overlap
    return logits_sum / counts.unsqueeze(0)

@torch.inference_mode()
def test(
    model,
    valid_loader,
    criterion,
    device,
    save_predictions=False,
    save_dir=None,
    test_data=None,
    save_metrics=True
):
    model.eval()

    if criterion.reduction != "mean":
        raise ValueError("This implementation expects reduction='mean'.")

    save_dir = run_results_dir / "test" if save_dir is None else Path(save_dir)

    if save_predictions:
        if test_data is None:
            raise ValueError("Provide test_data to save georeferenced predictions.")

        if not hasattr(valid_loader.dataset, "data_list"):
            raise ValueError("Expected a dataset with data_list.")

        if list(test_data) != list(valid_loader.dataset.data_list):
            raise ValueError("test_data must match the dataset order.")

        if valid_loader.batch_size != 1:
            raise ValueError("Use batch_size=1 to save predictions.")

        if not isinstance(
            valid_loader.sampler,
            torch.utils.data.SequentialSampler
        ):
            raise ValueError("Use shuffle=False when saving predictions.")

    if save_predictions or save_metrics:
        save_dir.mkdir(parents=True, exist_ok=True)

    total_loss = 0.0
    total_weight = 0.0

    TP = FP = FN = TN = 0
    sample_idx = 0
    per_chip_metrics = []

    for images, masks in valid_loader:

        for image, mask in zip(images, masks):

            mask = mask.cpu().long()
            valid = mask != IGNORE_INDEX
            n_valid = valid.sum().item()

            # Full-image sliding-window inference
            logits = sliding_window_logits(
                model,
                image,
                device,
                tile_size=CROP_SIZE,
                stride=None,
                tile_batch_size=TILE_BATCH_SIZE
            )

            prediction = logits.argmax(dim=0)

            # Save georeferenced prediction
            if save_predictions:

                x_path, y_path = test_data[sample_idx]
                x_path = Path(x_path)

                output_path = (
                    save_dir / f"{x_path.stem}_prediction.tif"
                )

                with rasterio.open(y_path) as src:
                    profile = src.profile.copy()

                profile.update(
                    driver="GTiff",
                    count=1,
                    dtype="uint8",
                    nodata=None
                )

                pred_np = prediction.numpy().astype(np.uint8)

                with rasterio.open(output_path, "w", **profile) as dst:
                    dst.write(pred_np, 1)

            sample_idx += 1

            chip_id = (Path(test_data[sample_idx - 1][0]).stem.replace("_S2Hand", "")
                       if test_data is not None else f"sample_{sample_idx-1}")
            per_chip_metrics.append({"chip_id": chip_id, "region": chip_id.rsplit("_", 1)[0],
                **binary_metrics(*confusion_counts(prediction.numpy(), mask.numpy()))})

            # Ignore samples without any valid ground-truth pixels
            if n_valid == 0:
                continue

            # Compute loss on the criterion's device
            logits_device = logits.unsqueeze(0).to(device)
            mask_device = mask.unsqueeze(0).to(device)

            loss = criterion(logits_device, mask_device)

            # Correct aggregation for weighted/unweighted CE
            if criterion.weight is not None:
                image_weight = criterion.weight[
                    mask_device[mask_device != IGNORE_INDEX]
                ].sum().item()
            else:
                image_weight = n_valid

            total_loss += loss.item() * image_weight
            total_weight += image_weight

            # Confusion matrix for the water class
            TP += (
                (prediction == 1) & (mask == 1) & valid
            ).sum().item()

            FP += (
                (prediction == 1) & (mask == 0) & valid
            ).sum().item()

            FN += (
                (prediction == 0) & (mask == 1) & valid
            ).sum().item()

            TN += (
                (prediction == 0) & (mask == 0) & valid
            ).sum().item()


    if total_weight == 0:
        raise RuntimeError("No valid test pixels.")

    metrics = {"test_loss": total_loss / total_weight, **binary_metrics(TP, FP, FN, TN)}

    if save_metrics:
        save_per_chip_metrics(per_chip_metrics, save_dir)
        # Use null for undefined metrics in standards-compliant JSON
        json_metrics = {
            k: (
                None
                if isinstance(v, float) and not math.isfinite(v)
                else v
            )
            for k, v in metrics.items()
        }

        with open(save_dir / "metrics.json", "w") as f:
            json.dump(json_metrics, f, indent=4)

    return metrics

In [ ]:
# A new directory is created each time this run cell is executed.
run_id = datetime.now().astimezone().strftime("%Y-%m-%d_%H-%M-%S_%f") + "_" + uuid4().hex[:8]
run_results_dir = Path("runs") / EXPERIMENT_NAME / f"test_{checkpoint_path.stem}_{run_id}"
run_results_dir.mkdir(parents=True, exist_ok=False)
print(f"Run directory: {run_results_dir}")

with (run_results_dir / "run.json").open("w", encoding="utf-8") as run_file:
    json.dump({
        "experiment": EXPERIMENT_NAME,
        "model_name": checkpoint_path.stem,
        "phase": "test",
        "run_id": run_id,
        "started_at": datetime.now().astimezone().isoformat(),
        "notebook": NOTEBOOK_PATH,
        "pretrained_weights": pretrained_path,
        "schema": SCHEMA,
        "pangaea_revision": PANGAEA_REVISION,
        "modality": "both",
        "split_counts": {k: len(v) for k, v in all_splits.items()},
        "inference_grid": "linspace_ceil_image_div_tile",
        "head_config": checkpoint["head_config"],
        "preprocessing": PREPROCESSING,
        "checkpoint_epoch": checkpoint["epoch"],
    "checkpoint": checkpoint_path.as_posix(),
    }, run_file, indent=4)

metrics_nonbolivia = test(
    model=model,
    valid_loader=test_loader_non_bolivia,
    criterion=criterion,
    device=device,
    save_predictions=True,
    save_dir=run_results_dir / "non_bol",
    test_data=test_data_non_bolivia
)

print("Non Bolivia test: ",metrics_nonbolivia)

In [ ]:
metrics_bolivia = test(
    model=model,
    valid_loader=test_loader_bolivia,
    criterion=criterion,
    device=device,
    save_predictions=True,
    save_dir=run_results_dir / "bol",
    test_data=test_data_bolivia
)

print("Bolivia test: ", metrics_bolivia)

In [ ]:
def aw_pw_fw_targets(
    path_to_labels,
    path_to_predictions,
    path_to_permanent_water,
):
    labels_dir = Path(path_to_labels)
    predictions_dir = Path(path_to_predictions)
    permanent_dir = Path(path_to_permanent_water)

    suffix = "_S2Hand_prediction.tif"
    prediction_files = sorted(predictions_dir.glob(f"*{suffix}"))

    if not prediction_files:
        raise FileNotFoundError(
            f"No prediction files found in {predictions_dir}"
        )

    for prediction_path in prediction_files:
        chip_id = prediction_path.name.removesuffix(suffix)

        label_path = labels_dir / f"{chip_id}_LabelHand.tif"
        permanent_path = permanent_dir / f"{chip_id}_JRCWaterHand.tif"

        for path in (label_path, permanent_path):
            if not path.is_file():
                raise FileNotFoundError(path)

        with (
            rasterio.open(prediction_path) as pred_src,
            rasterio.open(label_path) as label_src,
            rasterio.open(permanent_path) as perm_src,
        ):
            # Allow tiny floating-point differences in georeferencing.
            for src in (pred_src, perm_src):
                if (
                    src.shape != label_src.shape
                    or src.crs != label_src.crs
                    or not src.transform.almost_equals(
                        label_src.transform, precision=1e-10
                    )
                ):
                    raise ValueError(f"Raster grid mismatch: {chip_id}")

            prediction = pred_src.read(1)
            y = label_src.read(1).astype(np.int16)
            p = perm_src.read(1)

        y[y == -1] = 255

        if not np.isin(prediction, [0, 1]).all():
            raise ValueError(f"Non-binary prediction: {chip_id}")
        if not np.isin(y, [0, 1, 255]).all():
            raise ValueError(f"Unexpected hand labels: {chip_id}")
        if not np.isin(p, [0, 1]).all():
            raise ValueError(f"Unexpected JRC values: {chip_id}")

        targets = {
            "AW": y.copy(),
            "PW": y.copy(),
            "FW": y.copy(),
        }

        targets["PW"][(y == 1) & (p == 0)] = 255
        targets["FW"][(y == 1) & (p == 1)] = 255

        # Historical rule: exclude chips whose water positives
        # disappear after category masking. Retain originally dry chips.
        originally_has_water = np.any(y == 1)
        included = {
            category: not (
                category != "AW"
                and originally_has_water
                and not np.any(target == 1)
            )
            for category, target in targets.items()
        }

        yield {
            "chip_id": chip_id,
            "prediction": prediction,
            "targets": targets,
            "included": included,
        }

In [ ]:
hand_dir = Path(
    "dataset/sen1floods11_hand_labeled_only/HandLabeled"
)

rows = []

for split in ("non_bol", "bol"):
    records = aw_pw_fw_targets(
        path_to_labels=hand_dir / "LabelHand",
        path_to_predictions=run_results_dir / split,
        path_to_permanent_water=hand_dir / "JRCWaterHand",
    )

    for record in records:
        prediction = record["prediction"]

        for category, target in record["targets"].items():
            if not record["included"][category]:
                continue

            valid = target != 255

            tp = np.count_nonzero(valid & (prediction == 1) & (target == 1))
            fp = np.count_nonzero(valid & (prediction == 1) & (target == 0))
            fn = np.count_nonzero(valid & (prediction == 0) & (target == 1))
            tn = np.count_nonzero(valid & (prediction == 0) & (target == 0))

            rows.append({
                "split": split,
                "category": category,
                "chip_id": record["chip_id"],
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "tn": tn,
                "chip_iou": (tp + 1e-7) / (tp + fp + fn + 1e-7),
            })

per_chip = pd.DataFrame(rows)

summary = (
    per_chip.groupby(["split", "category"], sort=False)
    .agg(
        evaluated_chips=("chip_id", "size"),
        mean_chip_iou=("chip_iou", "mean"),
        tp=("tp", "sum"),
        fp=("fp", "sum"),
        fn=("fn", "sum"),
        tn=("tn", "sum"),
    )
    .reset_index()
)

union = summary["tp"] + summary["fp"] + summary["fn"]
summary["total_iou"] = summary["tp"] / union.replace(0, np.nan)

display(summary[
    ["split", "category", "evaluated_chips", "mean_chip_iou", "total_iou"]
])

per_chip.to_csv(run_results_dir / "croma_aw_pw_fw_per_chip.csv", index=False)
summary.to_csv(run_results_dir / "croma_aw_pw_fw_summary.csv", index=False)

In [ ]:
from IPython.display import display, Markdown

for split, title in [
    ("non_bol", "Table 1. Non-Bolivia test set"),
    ("bol", "Table 2. Bolivia test set"),
]:
    scores = (
        summary.loc[summary["split"].eq(split)]
        .set_index("category")["mean_chip_iou"]
        .reindex(["PW", "FW", "AW"])
    )

    table = pd.DataFrame(
        [scores.to_numpy()],
        columns=["PW", "FW", "AW"],
        index=["CROMA + UPerNet"],
    )
    table.index.name = "Model"

    display(Markdown(f"**{title} — Mean water IoU**"))
    display(table.style.format("{:.4f}"))

In [ ]:
import sen1floods11_visualization as visualization
# Reuse the existing plots with optical prediction filenames.
visualization.PREDICTION_SUFFIX = "_S2Hand_prediction.tif"
plot_predictions = visualization.plot_predictions

fig, axes, chip_ids = plot_predictions(
    predictions_dir=run_results_dir / "non_bol",
    hand_dir="dataset/sen1floods11_hand_labeled_only/HandLabeled",
    n=90,
    seed=42,
    save_path=run_results_dir / "croma_visualization_90.png",
    show_errors=True
)

plt.show()

In [ ]:

fig, axes, chip_ids = plot_predictions(
    predictions_dir=run_results_dir / "non_bol",
    hand_dir="dataset/sen1floods11_hand_labeled_only/HandLabeled",
    n=15,
    seed=42,
    save_path=run_results_dir / "croma_visualization_15.png",
    show_errors=True
)

plt.show()

In [ ]:
# Recompute from confusion counts so this also works with previously saved metric dictionaries.
for metrics in (metrics_nonbolivia, metrics_bolivia):
    metrics.update(binary_metrics(*(metrics[key] for key in ("TP", "FP", "FN", "TN"))))
# Water precision/recall use water as the positive class; commission = FP/(TP+FP).
all_metrics = metric_table({"non_bol": metrics_nonbolivia, "bol": metrics_bolivia}, run_results_dir)
display(all_metrics.T)
all_metrics[["accuracy", "water_iou", "miou", "precision", "recall", "f1",
             "omission_error", "commission_error"]].plot.bar(figsize=(13, 5), ylim=(0, 1))
plt.ylabel("Fraction")
plt.tight_layout()
plt.savefig(run_results_dir / "all_metrics_comparison.png", dpi=160)
plt.show()

# Report the original split above and the usable-paired-input subset separately.
usable_rows = []
for split, pairs in [("non_bol", test_data_non_bolivia), ("bol", test_data_bolivia)]:
    totals = np.zeros(4, dtype=np.int64)
    excluded = []
    for x_path, y_path in pairs:
        raw, label = load_image_pair((x_path, y_path))
        if not np.isfinite(raw).all(axis=0).any():
            excluded.append(x_path.name)
            continue
        with rasterio.open(run_results_dir / split / (x_path.stem + "_prediction.tif")) as src:
            pred = src.read(1)
        valid = np.isin(label, [0, 1])
        totals += [np.count_nonzero(valid & (pred == p) & (label == t))
                   for p, t in [(1, 1), (1, 0), (0, 1), (0, 0)]]
    usable_rows.append({"split": split, "included_images": len(pairs)-len(excluded),
                        "excluded_missing_joint_images": len(excluded),
                        **binary_metrics(*totals)})
    (run_results_dir / split / "excluded_missing_joint.json").write_text(json.dumps(excluded, indent=2))
usable_summary = pd.DataFrame(usable_rows)
usable_summary.to_csv(run_results_dir / "usable_joint_summary.csv", index=False)
input_quality_comparison = metric_table({
    "non_bol_original": metrics_nonbolivia,
    "non_bol_usable_joint_images": usable_summary.set_index("split").loc["non_bol"].to_dict(),
    "bol_original": metrics_bolivia,
    "bol_usable_joint_images": usable_summary.set_index("split").loc["bol"].to_dict(),
}, run_results_dir / "input_quality_comparison")
display(input_quality_comparison.T)

# Compare original-split results with the latest available unimodal UPerNet evaluations.
comparison_frames = []
comparison_sources = {"S1S2": str(run_results_dir)}
joint_metrics = pd.read_csv(run_results_dir / "metrics_summary.csv")
joint_metrics["experiment"] = "S1S2"
comparison_frames.append(joint_metrics)
for label, schema in [("S1", "croma_sar_upernet_pangaea_v2"), ("S2", "croma_s2_upernet_pangaea_v1")]:
    candidates = []
    for manifest in Path("runs/croma_sen1floods11").glob("*/test_*/run.json"):
        meta = json.loads(manifest.read_text(encoding="utf-8"))
        metrics_path = manifest.parent / "metrics_summary.csv"
        if meta.get("schema") == schema and metrics_path.is_file():
            candidates.append((meta["started_at"], metrics_path))
    if candidates:
        metrics_path = max(candidates, key=lambda item: item[0])[1]
        frame = pd.read_csv(metrics_path)
        frame["experiment"] = label
        comparison_frames.append(frame)
        comparison_sources[label] = str(metrics_path.parent)
comparison = pd.concat(comparison_frames, ignore_index=True)
comparison.to_csv(run_results_dir / "s1_s2_joint_comparison.csv", index=False)
(run_results_dir / "comparison_sources.json").write_text(json.dumps(comparison_sources, indent=2))
display(comparison[["experiment", "split", "miou", "water_iou", "precision", "recall", "f1"]])
if "S2" in comparison.experiment.values:
    paired = comparison[comparison.experiment == "S1S2"].set_index("split").join(
        comparison[comparison.experiment == "S2"].set_index("split"), rsuffix="_s2")
    for key in ("TP", "FP", "FN", "TN"):
        if key not in paired: raise ValueError("Missing comparison confusion counts.")
    assert ((paired.TP + paired.FP + paired.FN + paired.TN) ==
            (paired.TP_s2 + paired.FP_s2 + paired.FN_s2 + paired.TN_s2)).all()
    deltas = 100 * (paired[["miou", "water_iou", "precision", "recall"]] -
                    paired[["miou_s2", "water_iou_s2", "precision_s2", "recall_s2"]].rename(columns=lambda c: c.removesuffix("_s2")))
    deltas.to_csv(run_results_dir / "joint_minus_s2_percentage_points.csv")
    print("Joint minus S2 (percentage points; a single-seed comparison):")
    display(deltas)
comparison.pivot(index="split", columns="experiment", values="miou").plot.bar(ylim=(0, 1), figsize=(8, 4), rot=0)
plt.ylabel("Pooled mIoU")
plt.tight_layout()
plt.savefig(run_results_dir / "s1_s2_joint_miou.png", dpi=160)
plt.show()
